# LesothoHomeAI Training Demo Walkthrough

This notebook explains the end-to-end training and refresh process for the project in a screenshot-friendly way.

Important:
- The real runnable entrypoints remain the Python scripts in `scripts/`.
- You can still run each training stage separately from the terminal.
- Use this notebook to show the story of the workflow, the command order, and where the outputs go.

## 1. Prerequisites

Project root:

```text
C:\Users\lepha\Documents\Codex\Real Estate System
```

Install requirements once:

```powershell
py -m pip install -r requirements.txt
```

Useful output folders:
- `generated/artifacts/scraping/`
- `generated/artifacts/curation/`
- `generated/artifacts/vision/`
- `generated/artifacts/nlp/`
- `generated/artifacts/recommendation/`

## 2. Refresh live property data

Use this when you want the latest real listing sample before retraining or demoing the system.

In [ ]:
scraping_commands = [
    "py scripts/run_scraper.py --live-limit 10 --include-rentals --max-images 3",
]
for command in scraping_commands:
    print(command)

Expected artifacts:
- `generated/artifacts/scraping/real_only_properties_raw.csv`
- `generated/artifacts/scraping/real_only_properties_cleaned.csv`
- `generated/artifacts/scraping/real_only_scrape_summary.json`

## 3. Prepare the modeling datasets

This stage cleans the listings, keeps the residential subset, and prepares the CNN-ready training data.

In [ ]:
dataset_commands = [
    "py scripts/prepare_modeling_dataset.py",
    "py scripts/prepare_house_label_review.py",
    "py scripts/apply_house_label_review.py",
]
for command in dataset_commands:
    print(command)

Expected artifacts:
- `generated/artifacts/curation/properties_residential_curated.csv`
- `generated/artifacts/curation/properties_residential_cnn_candidates.csv`
- `generated/artifacts/review/properties_house_reviewed.csv`
- `generated/artifacts/review/house_label_review_summary.json`

## 4. Train the main house vision support model

This model helps the system infer style, condition, environment, and other image-based support signals.

In [ ]:
vision_command = "py scripts/train_house_vision_model.py"
print(vision_command)

Expected artifacts:
- `generated/artifacts/vision/house_vision_multitask.pt`
- `generated/artifacts/vision/house_vision_metrics.json`
- `generated/artifacts/vision/house_vision_predictions.csv`

## 5. Train the grouped bedroom support model

This improves the bedroom signal used by matching and lecturer-facing evaluation.

In [ ]:
bedroom_commands = [
    "py scripts/train_house_bedroom_model.py",
    "py scripts/evaluate_bedroom_improvement.py",
]
for command in bedroom_commands:
    print(command)

Expected artifacts:
- `generated/artifacts/vision/house_bedroom_multitask.pt`
- `generated/artifacts/vision/house_bedroom_metrics.json`
- `generated/artifacts/vision/house_bedroom_comparison.csv`
- `generated/artifacts/vision/house_bedroom_comparison.json`

## 6. Train the residential property-type support model

This produces the property-type classifier used in the curated house pipeline and lecturer dashboard.

In [ ]:
property_type_command = "py scripts/train_residential_property_type_model.py"
print(property_type_command)

Expected artifacts:
- `generated/artifacts/vision/residential_property_type_multitask.pt`
- `generated/artifacts/vision/residential_property_type_metrics.json`
- `generated/artifacts/vision/residential_property_type_predictions.csv`

## 7. Evaluate the NLP module

This updates the bilingual vocabulary metrics and the query sanity checks used by the dashboard.

In [ ]:
nlp_command = "py scripts/evaluate_nlp_module.py"
print(nlp_command)

Expected artifacts:
- `generated/artifacts/nlp/house_nlp_metrics.json`
- `generated/artifacts/nlp/house_nlp_query_results.csv`

## 8. Refresh the recommendation and marketing demo outputs

This is the step to run when you want the lecturer-facing smart matching and campaign dashboards to show the newest outputs.

In [ ]:
recommendation_command = "py scripts/run_house_recommendation_demo.py --listing-intent sale --top-n 3 --clients 6"
print(recommendation_command)

Expected artifacts:
- `generated/artifacts/recommendation/house_recommendation_properties.csv`
- `generated/artifacts/recommendation/house_recommendation_matches.csv`
- `generated/artifacts/recommendation/house_recommendation_campaigns.csv`
- `generated/artifacts/recommendation/house_recommendation_metrics.json`
- `generated/artifacts/recommendation/house_recommendation_fusion_summary.json`
- `generated/artifacts/recommendation/house_recommendation_marketing_summary.json`

## 9. Full terminal sequence

You can still run the stages separately. The usual end-to-end order is:

```powershell
py scripts/run_scraper.py --live-limit 10 --include-rentals --max-images 3
py scripts/prepare_modeling_dataset.py
py scripts/prepare_house_label_review.py
py scripts/apply_house_label_review.py
py scripts/train_house_vision_model.py
py scripts/train_house_bedroom_model.py
py scripts/evaluate_bedroom_improvement.py
py scripts/train_residential_property_type_model.py
py scripts/evaluate_nlp_module.py
py scripts/run_house_recommendation_demo.py --listing-intent sale --top-n 3 --clients 6
```

That means:
- yes, the training and evaluation scripts can still be run separately in the terminal
- the notebook is for explaining and presenting the workflow, not replacing the scripts